In [1]:
import time
import csv
import os
from datetime import datetime

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# ---- CONFIGURATION ---- #
EMAIL = os.getenv("SPLASH_EMAIL", "your_email_here")
PASSWORD = os.getenv("SPLASH_PASSWORD", "your_password_here")
URL = "http://dashboard2.splash360.fr/account/login?returnUrl=%2F"
GECKO_PATH = os.getenv("GECKO_PATH", r"C:\path\to\geckodriver.exe")
OUTPUT_FILE = "resultats_chicken.csv"

# ---- LANCEMENT DU DRIVER ---- #
service = Service(executable_path=GECKO_PATH)
driver = webdriver.Firefox(service=service)
wait = WebDriverWait(driver, 60)
driver.get(URL)

# ---- CONNEXION ---- #
wait.until(EC.presence_of_element_located((By.ID, "email"))).send_keys(EMAIL)
driver.find_element(By.ID, "password").send_keys(PASSWORD)
driver.find_element(By.CSS_SELECTOR, "button.login-btn").click()

# ---- FONCTIONS UTILITAIRES ---- #
def get_text(xpath, clean_euro=False):
    try:
        value = driver.find_element(By.XPATH, xpath).text.strip()
        return value.replace("€", "").replace(",", ".") if clean_euro else value
    except:
        return ""

def get_val(label):
    return get_text(f'//h5[contains(text(), "{label}")]/following::h3[1]', clean_euro=True)

def get_cmd(title_value):
    return get_text(f'//h5[@title="{title_value}"]/following::h3[1]')

def get_panier():
    try:
        bloc = driver.find_element(
            By.XPATH,
            '//h5[contains(text(), "PANIER MOYEN")]/ancestor::div[contains(@class,"card-body")]'
        )
        emporter = livraison = sur_place = ""

        for line in bloc.text.splitlines():
            if "emporter" in line:
                emporter = line.split(":")[1].replace("€", "").replace(",", ".").strip()
            elif "Livraison" in line:
                livraison = line.split(":")[1].replace("€", "").replace(",", ".").strip()
            elif "Sur place" in line:
                sur_place = line.split(":")[1].replace("€", "").replace(",", ".").strip()

        return emporter, livraison, sur_place
    except:
        return "", "", ""

# ---- CRÉATION / AJOUT AU CSV ---- #
headers = [
    "Date", "CA_TTC", "CA_HT", "Cmd_Resto", "CA_Uber_TTC", "CA_Uber_HT", "Cmd_Uber",
    "Panier_Emporter", "Panier_Livraison", "Panier_SurPlace"
]

file_exists = os.path.exists(OUTPUT_FILE)

with open(OUTPUT_FILE, mode="a", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)

    if not file_exists:
        writer.writerow(headers)

    # ---- SCRAP MANUEL ---- #
    while True:
        user_input = input("\n\n Appuie sur Entrée pour scraper, ou tape 'q' pour quitter : ")
        if user_input.lower() == "q":
            break

        try:
            date_lue = driver.find_element(By.CSS_SELECTOR, 'input[name="daterange"]').get_attribute("value")
        except:
            date_lue = datetime.now().strftime("%Y-%m-%d")

        print("\n Date lue dans l'interface :", date_lue)

        time.sleep(2)

        row = [
            date_lue,
            get_val("CA RESTAURANT TTC"),
            get_val("CA RESTAURANT HT"),
            get_cmd("Nombre total de commandes restaurant"),
            get_val("CA UBER TTC"),
            get_val("CA UBER HT"),
            get_cmd("Nombre total de commandes UBER"),
            *get_panier()
        ]

        writer.writerow(row)
        print("✔️ Données enregistrées pour :", date_lue)

# ---- FERMETURE ---- #
driver.quit()
print("✅ Terminé.")